# 🏆 Notebook 05 — Synthèse et Recommandations

**Phase 7 du projet KCorp Scouting Tool**

Ce notebook final combine les résultats du **Talent Score** (probabilité de promotion) et du **Playstyle Clustering** (profil de jeu) pour fournir des recommandations activables pour le recrutement.

In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from IPython.display import display
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
METRICS_DIR = ROOT / 'reports' / 'metrics'

print('Imports OK')

Imports OK


---
## 1. Fusion des résultats

On joint le dataset des scores avec les résultats du clustering pour obtenir une vue complète par joueur.

In [3]:
# Chargement des deux datasets finaux
scores_df = pd.read_csv(METRICS_DIR / 'talent_scores_players.csv')
clusters_df = pd.read_csv(METRICS_DIR / 'clustering_results.csv')

# Clé de jointure unique par joueur/split
join_keys = ['playername', 'league', '_source_year', 'split', 'position', 'teamname']

final_df = scores_df.merge(clusters_df[join_keys + ['cluster_position', 'umap_x', 'umap_y']], 
                           on=join_keys, how='inner')

# Chargement des archétypes de clusters
with open(METRICS_DIR / 'cluster_profiles.json', encoding='utf-8') as f:
    profiles = json.load(f)

def get_archetype(row):
    pos = row['position']
    cluster_id = row['cluster_position']
    if pd.isna(cluster_id) or pos not in profiles:
        return 'Inconnu'
    for p in profiles[pos]:
        if p['cluster'] == cluster_id:
            return p['archetype']
    return 'Inconnu'

final_df['playstyle_archetype'] = final_df.apply(get_archetype, axis=1)

print(f'Dataset consolidé : {len(final_df)} joueurs/splits')
display(final_df[['playername', 'position', 'league', 'talent_score', 'playstyle_archetype']].head())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\bower\\Documents\\Scouting DataScience\\Scouting\\notebooks\\reports\\metrics\\talent_scores_players.csv'

---
## 2. Le "KCorp Top 10" (Overall ERL)

Les 10 joueurs les plus prometteurs du circuit ERL, toutes positions et ligues confondues, en 2024-2025.

In [ ]:
top10 = (
    final_df
    .sort_values('talent_score', ascending=False)
    .drop_duplicates(subset=['playername'])  # On garde le meilleur split du joueur
    .head(10)
    [['playername', 'position', 'league', 'teamname', 'talent_score', 'playstyle_archetype', 'promoted_to_lec']]
    .reset_index(drop=True)
)
top10.index += 1
display(top10)

---
## 3. Top Talents par Position

Une équipe cherche souvent à recruter pour un rôle spécifique. Voici les 3 pépites recommandées pour chaque rôle.

In [ ]:
for pos in ['top', 'jng', 'mid', 'bot', 'sup']:
    print(f'\n--- TOP 3 : {pos.upper()} ---')
    pos_top = (
        final_df[final_df['position'] == pos]
        .sort_values('talent_score', ascending=False)
        .drop_duplicates(subset=['playername'])
        .head(3)
        [['playername', 'league', 'talent_score', 'playstyle_archetype']]
        .reset_index(drop=True)
    )
    pos_top.index += 1
    display(pos_top)

---
## 4. Stratégie de recrutement par Style

Exemple d'utilisation métier : *"Nous cherchons un Midlaner avec un style 'High DPS' pour remplacer notre joueur partant, de préférence hors de la LFL car les buyouts sont trop chers."*

In [ ]:
# Recherche spécifique
target_pos = 'mid'
target_style = 'High DPS'
excluded_leagues = ['LFL']

search_results = (
    final_df[
        (final_df['position'] == target_pos) & 
        (final_df['playstyle_archetype'].str.contains(target_style)) & 
        (~final_df['league'].isin(excluded_leagues))
    ]
    .sort_values('talent_score', ascending=False)
    .drop_duplicates(subset=['playername'])
    .head(5)
    [['playername', 'league', 'teamname', 'talent_score', 'playstyle_archetype']]
    .reset_index(drop=True)
)
search_results.index += 1

print('Résultats de la requête scout :')
display(search_results)

---
## 5. Conclusion du Projet

Le KCorp Scouting Tool est désormais un pipeline ML fonctionnel de bout en bout :

1. **Acquisition automatisée** : Ingestion des stats depuis Oracle's Elixir et des transferts depuis Leaguepedia.
2. **Feature Engineering contextuel** : Calcul de Z-scores pour comparer les joueurs équitablement au sein de leur ligue.
3. **Talent Scoring (Supervisé)** : La Régression Logistique permet de scorer le potentiel LEC d'un joueur, en séparant nettement les promus (médiane 58) des non-promus (médiane 21).
4. **Playstyle Clustering (Non-Supervisé)** : K-Means révèle les archétypes dominants à chaque position, et la similarity search permet de dénicher des "clones" de joueurs connus.

Cet outil prouve que les données macro de League of Legends peuvent produire des signaux de recrutement forts, optimisant ainsi le travail de visionnage (VOD review) des scouts professionnels.